# Training_File_Optimizer.ipynb — STEP 2C Optimizer Exploration

**Branch git**: `Optimizer_Exploration` (NON `main`).

**Scopo**: testare strategie di ottimizzazione non-mainstream (Prodigy LR-free, batch=1, schedule-free) contro la baseline AdamW per rompere il plateau val~0.28 (P12).

**Differenze rispetto a `Training_File.ipynb` / `Training_File_Sweep.ipynb`**:
- Quei notebook restano **invariati** su `main`.
- Questo notebook esiste solo su `Optimizer_Exploration` per esperimenti isolati.
- Aggiunge una cella **DRY-RUN** (15 epoche × 1 step) per validare l'ambiente Azure prima del FULL.
- Supporta nuovo optimizer **Prodigy** (`--optimizer prodigy`) introdotto in `train.py` sul branch.
- Supporta nuove CLI flag `--max_steps_per_epoch`, `--val_batch_size`, `--scheduler none` per
  budget step preciso e validation veloce con batch=1.

**Sequenza celle**:
1. **Cella 0** — Bootstrap deps (matplotlib, nbstripout, prodigyopt) + git pull
2. **Cella 1** — Environment check: import, versioni, GPU/CPU, file critici
3. **Cella 2** — DRY-RUN (15 ep × 1 step, ~1-2 min): verifica pipeline end-to-end **NO smoke**, **NO results**
4. **Cella 3** — EXPERIMENT_PLAN (config head-to-head Prodigy vs AdamW + reps display)
5. **Cella 4** — RUN experiment: chiama `train.py` con grad-explosion safe-stop
6. **Cella 5** — Push results per-run (commit + pull + push)
7. **Cella 6** — Analisi comparativa: val curves Plan A vs Plan B

**Safety**: il path `max_inf_streak` di `train.py` già garantisce: detection grad=inf → abort dopo N consecutivi → `crash_model.pt` salvato + CSV flushati (try/finally). Niente da aggiungere lato notebook.

---

### ⚠️ Note operative importanti

**Reps implicite**: nel nostro setup PyTorch il concetto di "reps" dei training diffusivi (AnimaStandAlone, kohya_ss) è codificato come `epochs × max_steps_per_epoch`. La Cella 3 esporrà esplicitamente le "reps per window" come quantità derivata — vedi sezione 1 del piano `precious-giggling-balloon.md`.

**Prodigy warmup interno**: il parametro `d` di Prodigy parte da `1e-6` e cresce step-by-step (`safeguard_warmup=True`). I primi ~50 batch di Plan A avranno update effettivo ≈ 0. Non è un bug — aspettatevi un "piatto" iniziale di durata ~50 step nei grafici G10/O2.

**Step budget**: `n_windows_per_traj = (1200 − seq_len)/stride + 1`. Con `seq_len=50, stride=25` ogni traiettoria genera 47 windows. Ignorarlo (come faceva la prima versione del notebook) sottostima i step totali di ~50×. La Cella 3 ora calcola il valore corretto.

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap (deps install + nbstripout + git pull)
# ===========================================================
# Su Azure il kernel azureml_py38 di solito ha: torch, numpy, pandas.
# Spesso mancano: matplotlib (versioni recenti), nbstripout, prodigyopt.
# Installiamo TUTTO in modo idempotente ad ogni esecuzione del notebook.
import sys

print('📦 Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas
!{sys.executable} -m pip install --quiet nbstripout
!{sys.executable} -m pip install --quiet prodigyopt

# nbstripout: evita conflitti git sui notebook (strippa output prima del diff)
# Operazione idempotente — non duplica config se già attiva.
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   ✅ deps installed + nbstripout attivato')

# Sync con remote (essenziale per pull ultimi fix tra sessioni Azure)
print('\n🔄 git fetch + pull (branch Optimizer_Exploration):')
!git fetch origin
!git checkout Optimizer_Exploration 2>/dev/null || git checkout -b Optimizer_Exploration origin/Optimizer_Exploration
!git pull --no-rebase --no-edit origin Optimizer_Exploration

print('\n📍 Branch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 — ENVIRONMENT CHECK: verifica tutto il necessario
# ===========================================================
# Validazione completa prima della Cella 2 (dry-run). Se qui fallisce qualcosa,
# NON eseguire le celle successive — fix install prima.
import sys, platform, os
from pathlib import Path

print(f'🐍 Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'📂 Working dir:     {os.getcwd()}')

# ── Core imports ────────────────────────────────────────────
checks = []

def _try_import(modname, attr=None):
    try:
        m = __import__(modname)
        v = getattr(m, '__version__', '?')
        return True, v
    except Exception as e:
        return False, str(e)

for mod in ['torch', 'numpy', 'pandas', 'matplotlib', 'prodigyopt']:
    ok, info = _try_import(mod)
    icon = '✅' if ok else '❌'
    print(f'   {icon} {mod:<14} v{info}')
    checks.append((mod, ok))

# ── PyTorch device ───────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'\n🎮 CUDA:            ✅ {torch.cuda.get_device_name(0)} '
          f'({torch.cuda.device_count()} GPU)')
    print(f'   torch.version.cuda: {torch.version.cuda}')
else:
    print(f'\n🎮 CUDA:            ❌ non disponibile — training su CPU')

# ── File critici del progetto ────────────────────────────────
print('\n📁 File critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'core/neurons.py',
          'data/generator.py', 'utils/plot_diagnostics.py',
          'scripts/preflight.py']:
    exists = os.path.isfile(f)
    print(f'   {"✅" if exists else "❌"} {f}')
    checks.append((f, exists))

# ── Verifica che train.py supporti prodigy ───────────────────
import subprocess
res = subprocess.run([sys.executable, 'train.py', '--help'],
                     capture_output=True, text=True)
has_prodigy = 'prodigy' in (res.stdout + res.stderr)
print(f'\n🔧 train.py supporta --optimizer prodigy: '
      f'{"✅" if has_prodigy else "❌ (branch sbagliato? rifare Cella 0)"}')
checks.append(('prodigy CLI', has_prodigy))

# ── Verdetto finale ──────────────────────────────────────────
n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n✅ ENV CHECK PASSED — puoi eseguire Cella 2 (dry-run)')
else:
    print(f'\n❌ ENV CHECK FAILED — {n_fail} problemi. NON eseguire Cella 2 finchè non risolti.')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 — DRY-RUN (NON è uno smoke test!)
# ===========================================================
# Scopo: lanciare train.py con CONFIG MINIMA (15 epoche × 1 step ciascuna)
# per verificare che l'INTERO PERCORSO funzioni su Azure SENZA bloccarsi
# per librerie mancanti, file inesistenti, errori di shape, etc.
#
# Differenza con smoke (--smoke flag):
#   smoke   → 1 epoca × 100 traiettorie ≈ 2 min CPU
#   dry-run → 15 epoche × 1 step ≈ 1-2 min CPU (touches all epoch boundaries)
#
# Strategia per ottenere 1 step/epoca:
#   - n_train=2 trajectories, seq_len=50, stride=25 → ~94 windows totali
#   - batch_size=200 → tutte le windows in 1 batch → 1 step/epoca esatto
#   - 15 epochs × 1 step = 15 gradient updates totali
#
# STEP 2C: esercitiamo anche le nuove flag --max_steps_per_epoch=1 e
# --val_batch_size=200 per assicurarci che siano implementate correttamente
# lato train.py. max_steps_per_epoch=1 è ridondante (batch=200 già produce
# 1 step/ep) ma testa il cap.
#
# Cosa verifica:
#   ✓ generator dataset (genera 2+2 traj highway)
#   ✓ DataLoader + windowing
#   ✓ forward + backward + clip_grad
#   ✓ BatchCSVLogger flush (anche con pochi batch)
#   ✓ epoch boundary × 15 (val_epoch + scheduler step + best_model save)
#   ✓ checkpoint save (best_model.pt + last_model.pt)
#   ✓ plot_all su CSV minimali (G1-G13)
#   ✓ NUOVO: --max_steps_per_epoch viene rispettato
#   ✓ NUOVO: --val_batch_size separato funziona
#
# Cosa NON verifica:
#   ✗ convergenza (15 step sono troppo pochi)
#   ✗ qualità ottimizzazione
#
# Output: checkpoints/DRYRUN_envcheck/ (NON pushato, NON committato)
import subprocess, sys, os, shutil, datetime, time

dryrun_tag = 'DRYRUN_envcheck'
# Cleanup precedente dry-run per ripartire pulito
for d in [f'checkpoints/{dryrun_tag}', 'data/cache_dryrun.pt']:
    if os.path.exists(d):
        if os.path.isdir(d): shutil.rmtree(d)
        else: os.remove(d)
        print(f'   🧹 cleanup {d}')

cmd = [
    sys.executable, 'train.py',
    '--epochs',              '15',
    '--n_train',             '2',
    '--n_val',               '2',
    '--batch_size',          '200',     # assorbe tutte le windows in 1 batch
    '--val_batch_size',      '200',     # NUOVO: esercita la flag separata
    '--max_steps_per_epoch', '1',       # NUOVO: ridondante ma testa il cap
    '--seq_len',             '50',      # default — evita degenerazioni windowing
    '--scheduler',           'plateau', # no scheduler complesso per dry-run
    '--lr',                  '1e-3',
    '--optimizer',           'adam',    # default safe (NO prodigy nel dry-run)
    '--max_inf_streak',      '3',       # abort rapido se grad esplode
    '--early_stop_patience', '0',       # disabilita per touchare tutte 15 ep
    '--data_cache',          'data/cache_dryrun.pt',
    '--tag',                 dryrun_tag,
    '--scenario_mix',        'highway',
    '--cut_in_ratio',        '0.0',
    '--log_every',           '1',
]

print(f'🧪 DRY-RUN: {" ".join(cmd)}\n')
t0 = time.time()
res = subprocess.run(cmd, capture_output=False)
elapsed = time.time() - t0

# ── Verdetto ─────────────────────────────────────────────────
print('\n' + '=' * 60)
print(f'⏱️  Tempo: {elapsed:.1f}s')
if res.returncode != 0:
    print(f'❌ DRY-RUN FAILED (exit code {res.returncode})')
    print('   NON procedere con Cella 3/4 finchè non funziona.')
    raise SystemExit('Dry-run failed')

# Verifica artefatti attesi
expected = [
    f'checkpoints/{dryrun_tag}/training_log.csv',
    f'checkpoints/{dryrun_tag}/training_batch_log.csv',
    f'checkpoints/{dryrun_tag}/best_model.pt',
    f'checkpoints/{dryrun_tag}/last_model.pt',
    f'checkpoints/{dryrun_tag}/plots',
]
missing = [p for p in expected if not os.path.exists(p)]
if missing:
    print(f'⚠️  DRY-RUN exit 0 ma artefatti mancanti: {missing}')
    print('   Investigare prima di Cella 3/4.')
else:
    import glob, pandas as pd
    n_plots = len(glob.glob(f'checkpoints/{dryrun_tag}/plots/*.png'))
    # Verifica che il cap max_steps_per_epoch=1 sia stato rispettato
    bdf = pd.read_csv(f'checkpoints/{dryrun_tag}/training_batch_log.csv')
    n_batches = len(bdf)
    expected_batches = 15  # 15 epoche × 1 step
    cap_ok = n_batches == expected_batches
    print(f'✅ DRY-RUN PASSED — {n_plots} plot generati')
    print(f'   batch_log: {n_batches} righe '
          f'(atteso: {expected_batches}, cap funziona: {"✅" if cap_ok else "❌"})')
    print('   Pipeline end-to-end funzionante su Azure. Procedi con Cella 3.')
    print(f'\n💡 Per ispezionare: ls checkpoints/{dryrun_tag}/')

In [ ]:
# ===========================================================
# CELLA 3 — EXPERIMENT PLAN: Prodigy vs AdamW head-to-head
# ===========================================================
# Hypothesis test: il plateau val~0.28 (P12) è causato da local minima?
#   Plan A — Prodigy + batch=1 + lr=1.0 + no scheduler
#            (gradient noise = regolarizzatore implicito anti-LM)
#   Plan B — AdamW + batch=8 + OneCycleLR (warmup 30% built-in)
#            (mainstream SOTA control)
#
# Step budget identico (~2850-3000 gradient updates) → confronto pulito.
# n_train=1500 (uguale a STEP 2A/2B per coerenza), highway-only.
#
# ⚠️  EARLY STOPPING DISABILITATO IN ENTRAMBI I PLAN
# Motivo: per testare l'uscita da local minima (ipotesi P12) servono epoche
# di "non miglioramento" durante le quali il rumore esplora il landscape.
# Se patience=4 fermasse il run al 4° plateau, perderemmo l'eventuale exit
# all'epoca 5+. Quindi: patience=0 → il training gira per TUTTE le epoche
# (tranne grad-explosion che resta attivo via max_inf_streak).
#
# ⚠️  BUDGET STEP CAPATO via --max_steps_per_epoch
# Il windowing genera ~47 windows/traj con seq_len=50, stride=25.
# Senza cap, Plan A produrebbe 70500 step/ep (50× il target). Il cap garantisce
# budget preciso: Plan A=3000 step (200×15ep), Plan B=2850 step (190×15ep).
#
# ⚠️  ARCHITETTURA = BASELINE P9_S2A (h=32, r=8, ~864 params)
# Il sweep STEP 2B ha falsificato P9 (capacità): val_best stabile a ~0.28
# su 11× range hidden (32→128, Δ=1.3‰). Quindi usiamo il baseline per:
#   - isolare la variabile sotto test (ottimizzatore, non capacità)
#   - 3× più veloce in BPTT (params 864 vs 2752 per h=64)
#   - architettura target deployment PYNQ-Z1

import time, subprocess, sys, os, datetime

# ── Costanti fisiche del dataset (allineate a config.py) ──────
SIM_DURATION_S = 120.0    # config.py:121
DT             = 0.1      # config.py:63
# Numero windows per traiettoria con stride=seq_len//2 (vedi train.py:855)
def _windows_per_traj(seq_len):
    n_timesteps_traj = int(SIM_DURATION_S / DT)        # 1200
    stride = seq_len // 2                              # 25 per seq_len=50
    return (n_timesteps_traj - seq_len) // stride + 1  # 47 per default

# ── Defaults comuni ────────────────────────────────────────────
COMMON = {
    'n_train':       1500,
    'n_val':         300,
    'seq_len':       50,
    'scenario_mix':  'highway',
    'cut_in_ratio':  0.0,
    # Architettura: BASELINE P9_S2A. P9 (capacità) falsificato dal sweep STEP 2B.
    # ~864 params, deployment target PYNQ-Z1, 3× più veloce di h=64.
    'cf_hidden_size': 32,
    'cf_rank':        8,
    # PINN weights — gli stessi di STEP 2A/2B baseline
    'lambda_data':   1.0,
    'lambda_phys':   0.1,
    'lambda_ou':     0.05,
    'lambda_bc':     1.0,
    # Val decoupling — STEP 2C fix #1
    'val_batch_size': 64,           # disaccoppia val da train (Plan A batch=1)
    # Safety: grad-explosion resta attiva, ma early stop NO
    'max_inf_streak':       20,
    'early_stop_patience':  0,      # DISABILITATO — vedi commento header cella
    'early_stop_delta':     0.005,
}

# ── Plan A: Prodigy stack ──────────────────────────────────────
# Step budget: max_steps_per_epoch=200 × 15 epoche = 3000 step (15 val points)
# Era epochs=2/max_steps=1500 → solo 2 val points, troppo pochi per tracciare
# l'eventuale uscita da plateau. Stesso budget, 15× più granularità.
PLAN_A = {
    **COMMON,
    'tag':                 'P12_S2C_planA_prodigy_b1',
    'optimizer':           'prodigy',
    'lr':                  1.0,            # Prodigy convention: 1.0 stima iniziale
    'batch_size':          1,
    'scheduler':           'none',         # fix #2 — Prodigy auto-adapta
    'epochs':              15,             # era 2 → ora 15 val points
    'max_steps_per_epoch': 200,            # cap → 200×15 = 3000 step totali
}

# ── Plan B: AdamW SOTA stack ───────────────────────────────────
# Step budget: max_steps_per_epoch=190 × 15 epoche = 2850 step (15 val points)
# OneCycleLR con warmup 30% built-in (evita grad-spike iniziale — fix #3)
PLAN_B = {
    **COMMON,
    'tag':                 'P12_S2C_planB_adamw_b8',
    'optimizer':           'adamw',
    'lr':                  2e-3,
    'batch_size':          8,
    'scheduler':           'onecycle',     # fix #3 — warmup 30% built-in
    'max_lr':              2e-3,
    'epochs':              15,
    'max_steps_per_epoch': 190,            # cap → 2850 step totali
}

EXPERIMENT_PLAN = [PLAN_A, PLAN_B]

# Quale eseguire? Modifica RUN_PLANS per attivare/disattivare.
RUN_PLANS = ['A', 'B']     # ['A'] solo Prodigy, ['B'] solo AdamW, ['A','B'] entrambi

# ── Display: calcolo step + reps implicite (anti-bug step budget) ─────
def derive_budget(plan):
    """Calcola step e reps reali partendo dal windowing del dataset."""
    wpt          = _windows_per_traj(plan['seq_len'])
    n_windows    = plan['n_train'] * wpt
    # ceil division per batch_size
    natural_spe  = -(-n_windows // plan['batch_size'])
    cap          = plan.get('max_steps_per_epoch')
    eff_spe      = min(natural_spe, cap) if cap and cap > 0 else natural_spe
    total_steps  = eff_spe * plan['epochs']
    # Reps implicite per window (vedi piano §1)
    impl_reps    = total_steps * plan['batch_size'] / n_windows
    return n_windows, natural_spe, eff_spe, total_steps, impl_reps

print(f'📋 EXPERIMENT PLAN: {len(EXPERIMENT_PLAN)} configs')
print(f'⚠️  Early stopping DISABILITATO (patience=0) per testare uscita da local minima')
print(f'⚠️  Budget step capato via --max_steps_per_epoch (windowing genera molti più step)')
print(f'⚠️  Architettura BASELINE h=32, r=8 (P9 falsificato → no extra capacity)')
for letter, plan in zip(['A', 'B'], EXPERIMENT_PLAN):
    active = '✅' if letter in RUN_PLANS else '⏸️'
    n_w, nat_spe, eff_spe, total, reps = derive_budget(plan)
    print(f'\n  {active} Plan {letter}: {plan["tag"]}')
    print(f'     optimizer  = {plan["optimizer"]}  lr={plan["lr"]}')
    print(f'     batch_size = {plan["batch_size"]} (train)  {plan["val_batch_size"]} (val)')
    print(f'     epochs     = {plan["epochs"]}  (NO early stop)  → {plan["epochs"]} val points')
    print(f'     scheduler  = {plan["scheduler"]}')
    print(f'     ── Step budget ──')
    print(f'       windows totali (n_train × {_windows_per_traj(plan["seq_len"])}/traj) = {n_w:>7d}')
    print(f'       natural step/ep (no cap)                = {nat_spe:>7d}')
    print(f'       capped a max_steps_per_epoch={plan.get("max_steps_per_epoch", "-")}         = {eff_spe:>7d}')
    print(f'       × {plan["epochs"]} epoche                          → {total:>7d} step totali')
    print(f'       reps implicite per window               = {reps:.3f}')
    print(f'     safety     = max_inf_streak={plan["max_inf_streak"]} (grad-explosion abort attivo)')

In [ ]:
# ===========================================================
# CELLA 4 — RUN: lancia train.py per ogni plan attivo
# ===========================================================
# Per ogni plan: chiama train.py via subprocess. Il safe-stop su grad=inf
# (max_inf_streak) è già integrato in train.py — niente da gestire qui.
# Se un plan fallisce, prosegue con il successivo (continue-on-failure).
import subprocess, sys, time, datetime

def _build_cli(plan):
    """Converte plan dict in argv list per train.py."""
    args = [
        sys.executable, 'train.py',
        '--epochs',          str(plan['epochs']),
        '--n_train',         str(plan['n_train']),
        '--n_val',           str(plan['n_val']),
        '--batch_size',      str(plan['batch_size']),
        '--seq_len',         str(plan['seq_len']),
        '--scheduler',       plan['scheduler'],
        '--lr',              str(plan['lr']),
        '--optimizer',       plan['optimizer'],
        '--scenario_mix',    plan['scenario_mix'],
        '--cut_in_ratio',    str(plan['cut_in_ratio']),
        '--cf_hidden_size',  str(plan['cf_hidden_size']),
        '--cf_rank',         str(plan['cf_rank']),
        '--lambda_data',     str(plan['lambda_data']),
        '--lambda_phys',     str(plan['lambda_phys']),
        '--lambda_ou',       str(plan['lambda_ou']),
        '--lambda_bc',       str(plan['lambda_bc']),
        '--max_inf_streak',  str(plan['max_inf_streak']),
        '--early_stop_patience', str(plan['early_stop_patience']),
        '--early_stop_delta',    str(plan['early_stop_delta']),
        '--data_cache',      f'data/cache_{plan["n_train"]}_'
                             f'{plan["scenario_mix"]}_cut{plan["cut_in_ratio"]}.pt',
        '--tag',             plan['tag'],
    ]
    # STEP 2C — nuove flag (passate solo se attive)
    if plan.get('max_steps_per_epoch') and plan['max_steps_per_epoch'] > 0:
        args += ['--max_steps_per_epoch', str(plan['max_steps_per_epoch'])]
    if plan.get('val_batch_size') and plan['val_batch_size'] > 0:
        args += ['--val_batch_size', str(plan['val_batch_size'])]
    # Scheduler-specific flag
    if plan['scheduler'] == 'onecycle':
        args += ['--max_lr', str(plan['max_lr'])]
    elif plan['scheduler'] == 'cosine':
        args += ['--T0', str(plan['T0'])]
    # 'none' e 'plateau' non richiedono flag aggiuntive
    return args

results = []
for letter, plan in zip(['A', 'B'], EXPERIMENT_PLAN):
    if letter not in RUN_PLANS:
        print(f'\n⏸️  Skipping Plan {letter} ({plan["tag"]})')
        continue
    
    print('\n' + '=' * 78)
    print(f'🚀 Plan {letter}: {plan["tag"]}')
    print('=' * 78)
    cmd = _build_cli(plan)
    print(f'CLI: {" ".join(cmd[2:])}\n')
    
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail (exit {res.returncode})'
    
    print(f'\n⏱️  Plan {letter} terminato in {elapsed/60:.1f}min — status: {status}')
    results.append({
        'plan':    letter,
        'tag':     plan['tag'],
        'status':  status,
        'elapsed': elapsed,
    })

print('\n' + '=' * 78)
print('🏁 EXPERIMENT COMPLETATO')
print('=' * 78)
for r in results:
    print(f'   Plan {r["plan"]:<3} ({r["tag"]}): {r["status"]:<25} '
          f'{r["elapsed"]/60:.1f}min')

In [ ]:
# ===========================================================
# CELLA 5 — PUSH results su branch Optimizer_Exploration
# ===========================================================
# Copia checkpoints/<tag>/ → results/<tag>/, commit, pull, push.
# Resilient: anche su training fallito gli artefatti CSV/plots vanno comunque su git.
import os, glob, shutil, datetime, subprocess
import numpy as np, pandas as pd

def push_run(tag, plan):
    src = f'checkpoints/{tag}'
    dst = f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   ⚠️  {src} mancante — niente da pushare')
        return False
    
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')
    
    # Crash info (se presente crash_model.pt — grad explosion early-stop)
    if os.path.isfile(f'{src}/crash_model.pt'):
        epoch_csv = f'{src}/training_log.csv'
        crash = ['Grad explosion early-stop detected (vedi training_batch_log.csv)']
        if os.path.isfile(epoch_csv):
            edf = pd.read_csv(epoch_csv)
            if len(edf) > 0:
                crash.append(f'Last epoch: {len(edf)}')
                crash.append(f'Best val_loss: {edf["val_total"].min():.5f}')
        with open(f'{dst}/CRASH_INFO.txt', 'w') as f:
            f.write('\n'.join(crash) + '\n')
    
    # Metriche per commit msg
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    val_str = ''
    ep_csv = f'{dst}/training_log.csv'
    if os.path.isfile(ep_csv):
        edf = pd.read_csv(ep_csv)
        if len(edf) > 0:
            val_str = (f"Best val_loss: {edf['val_total'].min():.4f} "
                       f"(E{int(edf['val_total'].idxmin())+1}/{len(edf)})\n")
    inf_str = ''
    bt_csv = f'{dst}/training_batch_log.csv'
    if os.path.isfile(bt_csv):
        bdf = pd.read_csv(bt_csv)
        n_inf = int(bdf['is_inf_grad'].sum())
        inf_str = f"Batch totali: {len(bdf)}  |  inf_grad: {n_inf}\n"
    
    msg = (
        f"results (Optimizer_Exploration): {tag} ({ts})\n\n"
        f"Plan: optimizer={plan['optimizer']}, batch={plan['batch_size']}, "
        f"lr={plan['lr']}, scheduler={plan['scheduler']}, epochs={plan['epochs']}\n"
        f"Scenario: {plan['scenario_mix']}, cut_in={plan['cut_in_ratio']}\n"
        f"Capacity: hidden={plan['cf_hidden_size']}, rank={plan['cf_rank']}\n"
        f"{val_str}{inf_str}\n"
        f"Artefatti: CSV (per-epoca + per-batch) + plots G1-G13.\n"
        f"Branch Optimizer_Exploration — NON merge su main senza review.\n"
    )
    with open('/tmp/opt_commit_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    
    !git add {dst}
    !git commit -F /tmp/opt_commit_msg.txt
    !rm /tmp/opt_commit_msg.txt
    print(f'   🔄 git pull --no-rebase --no-edit origin Optimizer_Exploration')
    !git pull --no-rebase --no-edit origin Optimizer_Exploration
    print(f'   📤 git push origin Optimizer_Exploration')
    !git push origin Optimizer_Exploration
    return True

if not results:
    print('⏭️  Nessun run eseguito — esegui Cella 4 prima')
else:
    for r in results:
        plan = next(p for p in EXPERIMENT_PLAN if p['tag'] == r['tag'])
        print(f'\n📤 Push {r["tag"]}...')
        push_run(r['tag'], plan)

In [ ]:
# ===========================================================
# CELLA 6 — ANALISI comparativa Plan A vs Plan B
# ===========================================================
# Confronto val curves + spike rate + LR profile. Tutti i grafici per-run
# (G1-G13) sono già in checkpoints/<tag>/plots/ — qui costruiamo solo le
# overlay tra plan.
import os, json, datetime, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Carica dati per ogni plan presente in checkpoints/ o results/
loaded = []
for letter, plan in zip(['A', 'B'], EXPERIMENT_PLAN):
    tag = plan['tag']
    for base in [f'results/{tag}', f'checkpoints/{tag}']:
        ep_csv = f'{base}/training_log.csv'
        bt_csv = f'{base}/training_batch_log.csv'
        if os.path.isfile(ep_csv):
            ep = pd.read_csv(ep_csv)
            bt = pd.read_csv(bt_csv) if os.path.isfile(bt_csv) else None
            loaded.append({'letter': letter, 'tag': tag, 'plan': plan,
                           'epoch_df': ep, 'batch_df': bt, 'base': base})
            break

if not loaded:
    print('❌ Nessun risultato trovato. Esegui Cella 4 prima.')
else:
    out_dir = f'opt_plots/{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
    os.makedirs(out_dir, exist_ok=True)
    colors = {'A': '#1f77b4', 'B': '#ff7f0e'}
    
    # ── O1: val_total per epoca (overlay) ─────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    for d in loaded:
        ep = d['epoch_df']
        lbl = f'Plan {d["letter"]}: {d["plan"]["optimizer"]} b={d["plan"]["batch_size"]}'
        ax.plot(ep['epoch'], ep['val_total'], 'o-', linewidth=2, markersize=8,
                color=colors[d['letter']], label=lbl)
    ax.axhline(0.2802, ls='--', color='red', alpha=0.6,
               label='P9_S2A baseline (0.2802)')
    ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
    ax.set_title('O1 — val_total per epoca (Plan A vs B)')
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/O1_val_per_epoch.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    # ── O2: loss per batch (linee continue, dato che batch_size differisce) ──
    fig, ax = plt.subplots(figsize=(10, 5))
    for d in loaded:
        if d['batch_df'] is None: continue
        bt = d['batch_df']
        # x = step index (riga del batch_log)
        ax.plot(np.arange(len(bt)), bt['loss_total'], '-',
                color=colors[d['letter']], linewidth=0.8, alpha=0.6,
                label=f'Plan {d["letter"]}')
    ax.set_xlabel('step (batch_idx cumulativo)'); ax.set_ylabel('loss_total')
    ax.set_title('O2 — loss_total per step (Plan A vs B su step budget identico ~2500)')
    ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/O2_loss_per_step.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    # ── O3: gradient norm per step (Plan A noisier? Plan B smoother?) ──
    fig, ax = plt.subplots(figsize=(10, 5))
    for d in loaded:
        if d['batch_df'] is None: continue
        bt = d['batch_df']
        gn = bt['gn_total_preclip'].replace([np.inf, -np.inf], np.nan).dropna()
        ax.plot(gn.index, gn.values, '-', color=colors[d['letter']],
                linewidth=0.7, alpha=0.5, label=f'Plan {d["letter"]}')
    ax.axhline(1.0, ls='--', color='red', alpha=0.5, label='clip threshold (1.0)')
    ax.set_xlabel('step'); ax.set_ylabel('gn_total_preclip (log)')
    ax.set_title('O3 — gradient norm per step (noise pattern)')
    ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/O3_grad_per_step.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    # ── Verdetto ─────────────────────────────────────────────
    display(Markdown('### 🎯 Risultati'))
    for d in loaded:
        ep = d['epoch_df']
        best = ep['val_total'].min()
        be   = int(ep['val_total'].idxmin()) + 1
        print(f'   Plan {d["letter"]} ({d["plan"]["optimizer"]:<7} b={d["plan"]["batch_size"]}): '
              f'best val={best:.4f} @E{be}/{len(ep)}')
    
    print(f'\n💾 Plot comparativi salvati in: {out_dir}/')